# Модуль 11.5. Дизайн инструментов: чиним tools на моке Cognopolis

Этот ноутбук — практика к лекции [«Дизайн инструментов: что значит „хороший tool"»](https://itrubnikov.github.io/Train_of_Thought/docs/modules/11-5-tool-design/). В лекции вы увидели линзу; здесь вы чините tools руками.

Сквозной пример — игра [Cognopolis](https://kindomklaster.com): житель-агент ходит по карте 7×7, рубит деревья, копает камень, дерётся с гоблином, крафтит инструменты и несёт добычу домой — на домашней клетке `(0,0)` рюкзак сам ложится на склад (авто-банк). Игра **могла бы** отдать модели один god-tool `do_action(action, target)` — но отдаёт расщеплённые инструменты (`move_dir`, `gather`, `fight`, `craft`, `heal_at_temple`), и каждое такое решение что-то даёт модели. Мы соберём оба набора на чистом Python (без сети) и измерим разницу.

**Что вы получите на выходе:**

- мок-мир Cognopolis (грид 7×7, ресурсы, инвентарь, кулдаун) — чистый Python, без ключей и без сети;
- два набора tools — «плохой» (god-tool, cryptic errors, свободные строки) и «хороший» (split-инструменты по живому API, `Pydantic`-схемы, обучающие ошибки, идемпотентность);
- детерминированный харнесс, который гоняет скриптового «агента» по обоим наборам и считает разницу в успехе вызовов и в восстановлении после ошибки;
- мини-линтер по чек-листу A→H, который оценивает спецификацию tool в баллах — до и после правок.

**Карта ноутбука:**

- **Блок 1** — мок-мир + два набора tools.
- **Блок 2 (ядро, keyless)** — харнесс bad-vs-good + линтер A→H.
- **Блок 3 (опционально, нужен ключ)** — живая модель MiniMax-M3 против обоих наборов.
- **Блок 4 (опционально)** — «хорошие» tools против живого API `https://kindomklaster.com`.
- **Задачи** — переписать god-tool, добавить обучающую ошибку, сделать мутацию идемпотентной, перегнать линтер до целевого score.

:::note Главное про запуск
Ноутбук исполняется **целиком и без единого ключа** (`Run all`, без правок). Блоки 3 и 4 делают мягкий пропуск (soft-skip), если ключа или сети нет, — keyless-прогон остаётся зелёным. Активность для сдачи — в финальной секции «Задачи».
:::


## Подготовка окружения

Ставим единственную зависимость ядра — `pydantic` (схемы аргументов для «хороших» tools). В Colab и Kaggle он чаще всего уже стоит; строка ниже это просто гарантирует. `requests` нужен только опциональному Блоку 4 — если его нет, блок аккуратно пропустится.

Код ниже не печатает шумных логов установки — только финальную строку с версией. Ничего, кроме `pydantic`, для keyless-ядра не требуется: ни сети, ни ключей.


In [ ]:
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import pydantic  # noqa: F401
except Exception:
    _pip_install("pydantic")
    import pydantic  # noqa: F401

print("pydantic", pydantic.VERSION)


## Блок 1. Мок-мир Cognopolis и два набора tools

Сначала — мир. Чтобы пощупать дизайн tools, не нужен сервер: достаточно маленькой модели мира на чистом Python. Наш мок повторяет **ровно те правила**, по которым живёт настоящая игра:

- грид **7×7**, координаты `0..6` по обеим осям;
- раскладка из сидера игры (`server/config.py`): дом `(0,0)`, деревья `(1,1)` и `(4,2)`, камни `(2,4)` и `(5,5)`, гоблин `(3,3)`;
- движение — **8 фиксированных направлений** (стороны света + диагонали), один шаг за вызов, координаты сервер не принимает;
- `gather` и `fight` работают **на своей клетке**: сначала встань на узел или на врага;
- шаг на дом `(0,0)` — **авто-банк**: рюкзак сам пересыпается на склад, отдельного действия «сдать» в игре нет;
- рюкзак с лимитом **20**; казна с золотом — на лечение в Храме;
- **кулдаун 1 секунда** после каждого действия — повтор раньше времени отбивается ошибкой.

Мок сознательно берёт подмножество мира: один гоблин (без волка, огра и Подземья), а `craft` не проверяет здание-станцию. Всё остальное — имена, лимиты, правила, тексты ошибок — как в живом сервере: один и тот же мир мы дальше подставим под оба набора tools, и разница будет только в интерфейсе, не в правилах.


In [ ]:
import time
from dataclasses import dataclass, field
from typing import Optional

# --- раскладка мира (из сидера живой игры: server/config.py) --------------
GRID = 7
TILES = {}
for _y in range(GRID):
    for _x in range(GRID):
        TILES[(_x, _y)] = "empty"
TILES[(0, 0)] = "home"
TILES[(1, 1)] = "tree"
TILES[(4, 2)] = "tree"
TILES[(2, 4)] = "rock"
TILES[(5, 5)] = "rock"
TILES[(3, 3)] = "goblin"

# узел -> (предмет, навык). Точно как NODE в server/game.py
NODE = {"tree": ("wood", "woodcutting"), "rock": ("stone", "mining")}

# 8 направлений одного шага — как DIRECTIONS в SDK (D-069)
DIR_DELTA = {
    "north": (0, -1), "south": (0, 1), "east": (1, 0), "west": (-1, 0),
    "northeast": (1, -1), "northwest": (-1, -1),
    "southeast": (1, 1), "southwest": (-1, 1),
}

# рецепты — из RECIPE_CATALOG живой игры (inputs тратятся со склада)
RECIPES = {
    "axe_handle": {"inputs": {"wood": 3}, "building": "sawmill"},
    "axe":        {"inputs": {"axe_handle": 1, "stone": 2}, "building": "sawmill"},
    "pickaxe":    {"inputs": {"axe_handle": 1, "stone": 4}, "building": "sawmill"},
}

INVENTORY_CAP = 20
COOLDOWN_SECONDS = 1.0
MAX_HP = 20
TEMPLE_HEAL_RATE = 1     # золота за 1 hp — как TEMPLE_HEAL_RATE живого сервера


@dataclass
class World:
    """Мок мира Cognopolis. Одно состояние, которое меняют tools."""
    x: int = 0
    y: int = 0
    hp: int = MAX_HP
    gold: int = 12                                  # казна (на Храм)
    inventory: dict = field(default_factory=dict)   # рюкзак (носимое)
    stored: dict = field(default_factory=dict)      # склад (после авто-банка)
    goblin_alive: bool = True
    busy_until: float = 0.0  # отметка времени конца кулдауна

    def in_bounds(self, x, y):
        return 0 <= x < GRID and 0 <= y < GRID

    def on_tile(self, content):
        """Стоит ли житель на клетке с таким content (D-069: добыча и бой — на своей клетке)."""
        return TILES.get((self.x, self.y)) == content

    def carried_total(self):
        return sum(self.inventory.values())

    def auto_bank(self):
        """Авто-банк дома (правило МИРА, не интерфейса): весь рюкзак -> склад. Возвращает,
        что легло на склад ({} если пересыпать нечего)."""
        banked = dict(self.inventory)
        for it, q in banked.items():
            self.stored[it] = self.stored.get(it, 0) + q
        self.inventory = {}
        return banked


def fresh_world():
    return World()

print("Мир собран:", GRID, "x", GRID, "тайлов; старт на (0,0); рюкзак-лимит", INVENTORY_CAP)


### Набор «плохой»: god-tool `do_action`

Теперь первый интерфейс к этому миру — тот, который игра намеренно **не** делает. Один инструмент на всё: `do_action(action, target)`, обе строки свободные. Ошибки — `{"error": "invalid"}`, без подсказок. Это соломенное чучело из лекции, собранное в коде.

Обратите внимание: модель, глядя на сигнатуру `do_action(action: str, target: str)`, не знает ни допустимых глаголов (`gather`? `cut`? `chop`?), ни как ходить (координатами? словом? куда?). Внутри — спрятанный `switch` по строке, который модель должна реконструировать вслепую. Это анти-паттерн A (god-tool), C (свободные строки) и E (cryptic error) разом.


In [ ]:
def do_action(world: World, action: str, target: str = "") -> dict:
    """Выполнить действие в игре.

    Единственный god-tool «плохого» набора. Обе строки свободные, ошибки немые.
    Это намеренно плохой дизайн — антигерой для контраста.
    """
    # кулдаун есть, но «плохой» tool про него молчит — отдаёт тот же немой invalid
    if time.monotonic() < world.busy_until:
        return {"error": "invalid"}

    # действие зашито в свободную строку: реконструируем switch вслепую
    if action == "move":
        # target вида "x,y" — ещё одна свободная строка, которую надо угадать
        try:
            tx, ty = [int(v) for v in target.split(",")]
        except Exception:
            return {"error": "invalid"}
        if not world.in_bounds(tx, ty):
            return {"error": "invalid"}
        if max(abs(world.x - tx), abs(world.y - ty)) != 1:   # один шаг, диагонали легальны
            return {"error": "invalid"}
        world.x, world.y = tx, ty
        if (world.x, world.y) == (0, 0):
            world.auto_bank()          # правило мира сработало... но ответ о нём молчит
        world.busy_until = time.monotonic() + COOLDOWN_SECONDS
        return {"ok": True}

    if action == "gather":
        if not (world.on_tile("tree") or world.on_tile("rock")):
            return {"error": "invalid"}
        if world.carried_total() >= INVENTORY_CAP:
            return {"error": "invalid"}
        item, _skill = NODE[TILES[(world.x, world.y)]]
        world.inventory[item] = world.inventory.get(item, 0) + 1
        world.busy_until = time.monotonic() + COOLDOWN_SECONDS
        return {"ok": True}

    if action == "fight":
        if not (world.goblin_alive and world.on_tile("goblin")):
            return {"error": "invalid"}
        world.goblin_alive = False
        world.busy_until = time.monotonic() + COOLDOWN_SECONDS
        return {"ok": True}

    # неизвестный глагол — и снова немой отказ
    return {"error": "invalid"}


BAD_TOOLS = {"do_action": do_action}
print("Плохой набор:", list(BAD_TOOLS))


Убедимся, что «плохой» tool действительно ставит модель в положение угадывания. Прогоним пару наивных вызовов — таких, какие сделала бы модель, впервые увидевшая `do_action`.

Что вы увидите: пара промахов на угадывании глагола, промах `gather` на пустой клетке — и рабочий путь, найденный вслепую. Каждый отказ отвечает одинаковым немым `{"error": "invalid"}` — модель не понимает, что не так: глагол? цель? позиция?


In [ ]:
w = fresh_world()
# модель хочет срубить дерево и угадывает глаголы/цели
print("cut/big tree ->", do_action(w, "cut", "big tree"))      # неизвестный глагол
print("chop/tree    ->", do_action(w, "chop", "tree"))          # синоним — тоже мимо
print("gather (клетка пустая) ->", do_action(w, "gather", ""))  # глагол верный, но узла под ногами нет
# дойдём до дерева на (1,1): из (0,0) это ОДИН диагональный шаг
print("move 1,1 ->", do_action(w, "move", "1,1"))
time.sleep(COOLDOWN_SECONDS)   # ждём кулдаун честно
print("gather (у дерева) ->", do_action(w, "gather", ""), "| рюкзак:", w.inventory)


Видите главную боль: каждый отказ — это `{"error": "invalid"}` без единой подсказки. На неизвестном глаголе, на пустой клетке, на неверном формате `target` — ответ один и тот же. Для агента это тупик: единственная стратегия — перебирать синонимы и координаты вслепую. Именно так «тупит» агент из Модуля 10.

### Набор «хороший»: split-инструменты по живому API

Теперь — то, что делает настоящая Cognopolis. Инструменты по одной цели, `Pydantic`-схемы как рельсы для аргументов, обучающие ошибки с `code` и подсказкой, кулдаун как защита от дубля на ретрае.

Сначала схемы аргументов. `Pydantic`-модель — это та же `JSON Schema`, которую SDK отдаёт модели: типы и допустимые значения видны заранее, вызов проверяется **до** вашей логики. Главный `enum` живого API — `direction` из восьми направлений: сервер вообще не принимает координаты, поэтому «шаг в несоседнюю клетку» невозможен по построению. Второй — необязательный `resource` у `gather`: страховка-ожидание, а не свободная строка.


In [ ]:
from enum import Enum
from pydantic import BaseModel

class Direction(str, Enum):
    north = "north"; south = "south"; east = "east"; west = "west"
    northeast = "northeast"; northwest = "northwest"
    southeast = "southeast"; southwest = "southwest"

class Resource(str, Enum):
    wood = "wood"
    stone = "stone"

class MoveArgs(BaseModel):
    direction: Direction            # enum из 8 значений — мимо не пройти
    reason: Optional[str] = None    # необязательное «зачем» — журнал намерений

class GatherArgs(BaseModel):
    # необязательная страховка-ожидание: None -> взять узел под ногами (как в реальном gather)
    resource: Optional[Resource] = None
    reason: Optional[str] = None

class CraftArgs(BaseModel):
    recipe: str                     # проверяется логикой против каталога RECIPES
    reason: Optional[str] = None

class ActionArgs(BaseModel):
    reason: Optional[str] = None    # тело для fight/heal_at_temple — только reason

# Покажем, что Pydantic ловит «не то слово» сам, до игровой логики:
for bad_call in (lambda: MoveArgs(direction="up"),
                 lambda: GatherArgs(resource="timber")):
    try:
        bad_call()
    except Exception as e:
        print("Pydantic отбил недопустимое значение — это и есть рельсы:")
        print("  ", str(e).splitlines()[0])

print("Допустимые direction:", [d.value for d in Direction])
print("Допустимые resource:", [r.value for r in Resource])


Теперь обучающие ошибки. В реальной игре каждое нарушение правил — это `GameError(code, message)`, и каждая ошибка закреплена тестом с говорящим именем (`..._is_explained`). Соберём ту же машинку: маленький конструктор ошибки, который всегда возвращает `code` (для ветвления в коде) и человекочитаемый `message` (что не так и что делать).

Строки сообщений ниже — **дословно** из живого сервера (`server/errors.py`). Это не наш пересказ, а тот самый текст, который модель читает и по которому чинится за один шаг.


In [ ]:
def err(code: str, message: str) -> dict:
    """Конверт обучающей ошибки: машиночитаемый code + человекочитаемый message."""
    return {"error": {"code": code, "message": message}}

# дословные сообщения живого Cognopolis (server/errors.py, проверены тестами)
MSG = {
    "no_resource_here":  "There is no resource node (tree/rock) on your tile — step onto one first.",
    "inventory_full":    "Рюкзак полон — вернись домой (0,0), там разгрузишься на склад сам.",
    "no_enemy_here":     "There is no living enemy on your tile — step onto one first.",
    "already_full_hp":   "hp is already full — nothing to heal at the Храм.",
}

def at_map_edge_msg(direction: str) -> str:
    # живой сервер называет направление, в котором упёрлись, — чтобы агент выбрал другое
    return f"You are at the edge of the map — cannot step {direction} any further."

def cooldown_msg(seconds_left: float) -> str:
    # точная формула живого сервера: f"Character is on cooldown for {x:.2f}s."
    return f"Character is on cooldown for {seconds_left:.2f}s."

def not_enough_resources_msg(need: dict, have: dict) -> str:
    short = ", ".join(f"{r}: need {q}, have {have.get(r, 0)}" for r, q in need.items())
    return f"Not enough stored resources ({short}). Gather more, then return home (0,0) to bank it."

def not_enough_gold_msg(need: int, have: int) -> str:
    return f"Not enough gold in the казна: need {need}, have {have}."

print("Сообщения ошибок загружены:", len(MSG), "+ шаблоны (at_map_edge, cooldown, not_enough_*)")


И наконец сами «хорошие» tools — та самая пятёрка из лекции: `move_dir`, `gather`, `fight`, `craft`, `heal_at_temple`. Каждый — про одну цель, проверяет свои предусловия и при нарушении возвращает обучающую ошибку. Все мутации защищены кулдауном: повтор раньше времени не дублирует эффект, а аккуратно отбивается `character_on_cooldown`.

Заметьте структуру успешного ответа — это конверт `{result, cooldown, character}` из лекции: `result` — что произошло (машиночитаемо), `cooldown` — через сколько можно снова, `character` — свежее состояние, чтобы модели не нужен был лишний вызов `get_character`. А у шага на дом в `result` появляется поле `banked` — авто-банк сообщает о себе сам, без лишнего чтения.

Одну ошибку мы **нарочно оставили полуслепой**: `craft` на неизвестный рецепт отвечает голым «No such recipe here.» — без подсказки, какие рецепты бывают. Это ваш полигон для Задачи 2.


In [ ]:
def _cooldown_left(world: World) -> float:
    return max(0.0, world.busy_until - time.monotonic())

def _character_view(world: World) -> dict:
    """Свежее состояние — кладём в каждый успешный ответ (поля под след. шаг)."""
    return {
        "x": world.x, "y": world.y, "hp": world.hp, "gold": world.gold,
        "inventory": dict(world.inventory), "stored": dict(world.stored),
        "inventory_cap": INVENTORY_CAP,
        "cooldown": round(_cooldown_left(world), 2),
    }

def _ok(world: World, result: dict) -> dict:
    """Единый конверт успеха, как у живого API."""
    world.busy_until = time.monotonic() + COOLDOWN_SECONDS
    return {"result": result, "cooldown": COOLDOWN_SECONDS, "character": _character_view(world)}

def _guard_cooldown(world: World):
    left = _cooldown_left(world)
    if left > 0:
        return err("character_on_cooldown", cooldown_msg(left))
    return None


def move_dir(world: World, direction: str, reason: Optional[str] = None) -> dict:
    """Шаг на одну клетку в сторону direction (8 направлений, D-069). Координат нет — прислать
    несоседнюю клетку невозможно по построению. Дом (0,0) — авто-банк, result несёт banked."""
    body = MoveArgs(direction=direction, reason=reason)   # рельсы: enum проверен здесь
    cd = _guard_cooldown(world)
    if cd:
        return cd
    dx, dy = DIR_DELTA[body.direction.value]
    nx, ny = world.x + dx, world.y + dy
    if not world.in_bounds(nx, ny):
        return err("at_map_edge", at_map_edge_msg(body.direction.value))
    world.x, world.y = nx, ny
    banked = world.auto_bank() if (nx, ny) == (0, 0) else {}
    return _ok(world, {"moved_to": {"x": nx, "y": ny}, "banked": banked})


def gather(world: World, resource: Optional[str] = None, reason: Optional[str] = None) -> dict:
    """Добыть ресурс с узла НА СВОЕЙ клетке. resource — необязательная страховка-ожидание:
    указали "wood", а стоите на камне -> честный no_resource_here вместо «добыл не то»."""
    body = GatherArgs(resource=resource, reason=reason)   # enum проверится здесь
    cd = _guard_cooldown(world)
    if cd:
        return cd
    content = TILES.get((world.x, world.y))
    if content not in NODE:
        return err("no_resource_here", MSG["no_resource_here"])
    item, skill = NODE[content]
    if body.resource is not None and body.resource.value != item:
        return err("no_resource_here", MSG["no_resource_here"])
    if world.carried_total() >= INVENTORY_CAP:
        return err("inventory_full", MSG["inventory_full"])
    world.inventory[item] = world.inventory.get(item, 0) + 1
    # форма result — как у живого gather: {gathered, amount, gather_bonus, skill}
    return _ok(world, {"gathered": item, "amount": 1, "gather_bonus": 0, "skill": skill})


def fight(world: World, reason: Optional[str] = None) -> dict:
    """Атаковать врага НА СВОЕЙ клетке. Пустая клетка и уже побеждённый враг неразличимы —
    один код no_enemy_here, как в живой игре (никакого оракула таймера респауна)."""
    ActionArgs(reason=reason)
    cd = _guard_cooldown(world)
    if cd:
        return cd
    if not (world.goblin_alive and world.on_tile("goblin")):
        return err("no_enemy_here", MSG["no_enemy_here"])
    world.goblin_alive = False
    return _ok(world, {"combat_log": {"outcome": "win", "enemy": "goblin", "xp_gained": 10}})


def craft(world: World, recipe: str, reason: Optional[str] = None) -> dict:
    """Скрафтить рецепт из каталога RECIPES; ингредиенты тратятся со СКЛАДА (сначала добудь
    и отнеси домой). Мок не проверяет здание-станцию (упрощение)."""
    body = CraftArgs(recipe=recipe, reason=reason)
    cd = _guard_cooldown(world)
    if cd:
        return cd
    spec = RECIPES.get(body.recipe)
    if spec is None:
        # НАРОЧНО полуслепая ошибка — почините в Задаче 2 (листинг каталога, как у живого)
        return err("unknown_recipe", "No such recipe here.")
    need = spec["inputs"]
    if any(world.stored.get(r, 0) < q for r, q in need.items()):
        return err("not_enough_resources", not_enough_resources_msg(need, world.stored))
    for r, q in need.items():
        world.stored[r] -= q
    world.stored[body.recipe] = world.stored.get(body.recipe, 0) + 1
    return _ok(world, {"crafted": body.recipe, "spent": dict(need)})


def heal_at_temple(world: World, reason: Optional[str] = None) -> dict:
    """Вылечиться до полного hp за золото: cost = (max_hp − hp) × TEMPLE_HEAL_RATE из казны."""
    ActionArgs(reason=reason)
    cd = _guard_cooldown(world)
    if cd:
        return cd
    missing = MAX_HP - world.hp
    if missing <= 0:
        return err("already_full_hp", MSG["already_full_hp"])
    cost = missing * TEMPLE_HEAL_RATE
    if world.gold < cost:
        return err("not_enough_gold", not_enough_gold_msg(cost, world.gold))
    world.gold -= cost
    world.hp = MAX_HP
    return _ok(world, {"healed": missing, "cost": cost, "hp": world.hp})


GOOD_TOOLS = {"move_dir": move_dir, "gather": gather, "fight": fight,
              "craft": craft, "heal_at_temple": heal_at_temple}
print("Хороший набор:", list(GOOD_TOOLS))


Сравним два набора на одном и том же промахе — «добыть на пустой клетке». Это самый показательный момент лекции: одинаковая ситуация, разные ответы.


In [ ]:
w_bad = fresh_world()
w_good = fresh_world()
print("ПЛОХОЙ  gather на пустой клетке:", do_action(w_bad, "gather", ""))
print("ХОРОШИЙ gather на пустой клетке:", gather(w_good))
print()
print("Из плохого ответа модель не знает, что не так.")
print("Из хорошего — знает: code=no_resource_here и текст 'step onto one first' -> следующий ход: move_dir к дереву.")


## Блок 2 (ядро, keyless). Харнесс bad-vs-good и линтер A→H

Это центр ноутбука и он работает без ключей. Две вещи:

1. **Детерминированный харнесс.** Скриптовый «агент» с фиксированной стратегией гоняет один и тот же сценарий — «дойти до дерева, добыть, принести домой (авто-банк)» — через оба набора tools. Мы считаем, сколько вызовов прошло успешно и сколько ошибок агент сумел исправить. Никакой LLM: стратегия зашита, чтобы цифры были воспроизводимы.

2. **Линтер A→H.** Маленький набор правил-как-код, который читает спецификацию tool (имя, описание, аргументы, поведение ошибок) и выставляет балл по восьми буквам чек-листа из лекции. Прогоним его на `do_action` и на `gather` — увидим разрыв в score.

Начнём с харнесса. Ключевая метрика — **error-recovery**: умеет ли агент, получив ошибку, сделать осмысленный следующий шаг. На «хорошем» наборе обучающая ошибка прямо подсказывает ход; на «плохом» — `{"error":"invalid"}` не подсказывает ничего, и «агенту» остаётся слепой перебор.


In [ ]:
RECOVERABLE = {"no_resource_here", "inventory_full", "at_map_edge",
               "character_on_cooldown", "unknown_recipe",
               "not_enough_resources", "not_enough_gold", "already_full_hp"}

def run_good_scenario(verbose=False):
    """Скриптовый агент на ХОРОШЕМ наборе: дойти до дерева (1,1), добыть, принести домой
    (авто-банк на (0,0)). Стратегия читает code ошибки и чинится за один шаг."""
    w = fresh_world()
    calls = 0
    ok = 0
    recovered = 0  # ошибок, после которых агент сделал осмысленный следующий шаг

    # дерево (1,1) из дома (0,0) — ОДИН диагональный шаг (D-069: диагонали легальны).
    plan = [
        ("gather", {}),                            # промах: клетка пустая -> агент поймёт по code
        ("move_dir", {"direction": "southeast"}),  # (0,0) -> (1,1), прямо на дерево
        ("gather", {"resource": "wood"}),          # страховка-ожидание: точно дерево
        ("gather", {}),                            # ещё одна древесина
        ("move_dir", {"direction": "northwest"}),  # (1,1) -> (0,0): дом, авто-банк -> banked
    ]
    for name, kwargs in plan:
        if calls > 0:
            time.sleep(_cooldown_left(w))
        resp = GOOD_TOOLS[name](w, **kwargs)
        calls += 1
        if "error" in resp:
            code_ = resp["error"]["code"]
            if verbose:
                print(f"  {name}{kwargs} -> ERROR {code_}: {resp['error']['message']}")
            if code_ in RECOVERABLE:
                recovered += 1
        else:
            ok += 1
            if verbose:
                print(f"  {name}{kwargs} -> OK {resp['result']}")
    return {"calls": calls, "ok": ok, "recovered": recovered,
            "stored": dict(w.stored)}

print("Хороший набор, сценарий 'добыть и принести домой':")
res_good = run_good_scenario(verbose=True)
print()
print("Итог good:", res_good)


Теперь тот же сценарий на «плохом» наборе. «Агент» здесь вынужден угадывать строки: глагол `do_action` и `target` вида `"x,y"`. И главное — когда он промахивается, ответ `{"error": "invalid"}` не несёт `code`, по которому можно ветвиться. Поэтому каждую ошибку мы засчитываем как **невосстановимую**: агенту нечего прочитать, остаётся слепой перебор.

Чтобы было честно, дадим «плохому» агенту ту же последовательность намерений — но через единственный god-tool и со свободными строками, как их прислала бы модель.


In [ ]:
def run_bad_scenario(verbose=False):
    """Скриптовый агент на ПЛОХОМ наборе: те же намерения, но через god-tool и свободные строки.
    Ошибки немые (нет code) -> восстановиться по ним нельзя."""
    w = fresh_world()
    calls = 0
    ok = 0
    recovered = 0

    # модель угадывает: сначала синонимы глагола, потом находит рабочий путь
    plan = [
        ("cut", "big tree"),     # угадывание глагола -> invalid
        ("chop", "tree"),        # ещё синоним -> invalid
        ("gather", ""),          # глагол угадан, но клетка пустая -> invalid (немой!)
        ("move", "1,1"),         # угадала формат "x,y" -> диагональный шаг на дерево
        ("gather", ""),
        ("gather", ""),
        ("move", "0,0"),         # домой; авто-банк сработал, но {"ok": true} о нём молчит
    ]
    for action, target in plan:
        if calls > 0:
            time.sleep(max(0.0, w.busy_until - time.monotonic()))
        resp = do_action(w, action, target)
        calls += 1
        if "error" in resp:
            if verbose:
                print(f"  do_action({action!r}, {target!r}) -> {resp}")
            # немая ошибка: нет code -> агент не может осмысленно починиться
            recovered += 0
        else:
            ok += 1
            if verbose:
                print(f"  do_action({action!r}, {target!r}) -> OK")
    return {"calls": calls, "ok": ok, "recovered": recovered,
            "stored": dict(w.stored)}

print("Плохой набор, те же намерения:")
res_bad = run_bad_scenario(verbose=True)
print()
print("Итог bad:", res_bad)


Сведём цифры в одну табличку. Смотрите не на абсолютные числа (они зависят от сценария), а на **разрыв**: на «хорошем» наборе доля успешных вызовов выше, а ошибки — восстановимые (агент знает следующий шаг). На «плохом» успех ниже, а каждая ошибка — тупик.

И одна деталь напоследок: оба агента донесли дерево до склада — авто-банк сработал в обоих мирах, это правило мира. Но «хороший» интерфейс **сообщил** об этом полем `banked` в ответе, а «плохой» промолчал (`{"ok": true}`): агент на плохом наборе даже не знает, что разгрузился. Дизайн ответа — тоже дизайн tools.


In [ ]:
def pct(a, b):
    return f"{(100.0 * a / b):.0f}%" if b else "n/a"

print(f"{'метрика':<28}{'плохой':>12}{'хороший':>12}")
print("-" * 52)
print(f"{'всего вызовов':<28}{res_bad['calls']:>12}{res_good['calls']:>12}")
print(f"{'успешных вызовов':<28}{res_bad['ok']:>12}{res_good['ok']:>12}")
print(f"{'доля успеха':<28}{pct(res_bad['ok'], res_bad['calls']):>12}{pct(res_good['ok'], res_good['calls']):>12}")
print(f"{'ошибок восстановлено':<28}{res_bad['recovered']:>12}{res_good['recovered']:>12}")
print(f"{'на складе (авто-банк)':<28}{str(res_bad['stored']):>12}{str(res_good['stored']):>12}")
print()
assert res_good["stored"].get("wood", 0) >= 2, "хороший агент должен донести дерево до склада"
assert res_bad["stored"].get("wood", 0) >= 2, "мир один: плохой агент тоже доносит (но не знает об этом)"
print("Оба агента добыли дерево, но 'хороший' восстанавливается после промахов и видит banked;")
print("'плохой' слеп и на ошибке, и на успехе.")


### Линтер A→H: правила как код

Чек-лист A→H из лекции — это не теория, его можно выразить кодом. Линтер ниже принимает **спецификацию tool** (словарь с именем, описанием, аргументами и флагами поведения) и начисляет по баллу за каждую выполненную букву. Это та же логика, по которой работает скилл `tool-design-reviewer`, только в миниатюре.

Восемь проверок:

- **A** имя — глагол+существительное в `snake_case`, не god-tool;
- **B** description — есть и не «худая» однострочная заглушка;
- **C** аргументы — есть `enum` вместо свободных строк там, где у значения закрытый список;
- **D** возврат — структурный (не проза), есть поля под следующий шаг;
- **E** ошибки — есть `code` и человекочитаемый `message`;
- **F** идемпотентность — мутация защищена (кулдаун/idempotency-key);
- **G** безопасность — узкий blast radius (нет `shell`/`eval`/произвольного http);
- **H** производительность — выдача ограничена (например, `limit`).


In [ ]:
GOD_NAMES = {"do_action", "do", "execute", "run", "action", "perform"}
DANGEROUS = {"shell", "run_code", "eval", "exec", "http_get_any", "http_get"}

def lint_tool(spec: dict) -> dict:
    """Оценить спецификацию tool по чек-листу A->H. Возвращает {score, max, checks, misses}."""
    checks = {}

    name = spec.get("name", "")
    # A: snake_case глагол+существительное, не god-tool
    checks["A_name"] = (
        bool(name)
        and name not in GOD_NAMES
        and name == name.lower()
        and not spec.get("free_string_action", False)
    )
    # B: description есть и это не «худая» заглушка
    desc = (spec.get("description") or "").strip()
    checks["B_description"] = len(desc) >= 30 and desc.lower() not in {
        "выполнить действие", "do something", "tool"}
    # C: enum вместо свободной строки там, где значение из закрытого списка
    args = spec.get("args", {})  # {arg: {"type": "...", "enum": bool}}
    free_strings = [a for a, m in args.items()
                    if m.get("type") == "str" and m.get("closed_set")
                    and not m.get("enum")]
    checks["C_args_enum"] = len(free_strings) == 0
    # D: структурный возврат с полями под следующий шаг
    checks["D_return_structured"] = (
        spec.get("returns_structured", False)
        and bool(spec.get("next_step_fields"))
    )
    # E: ошибки с code + message
    checks["E_errors_teach"] = (
        spec.get("error_has_code", False) and spec.get("error_has_message", False))
    # F: идемпотентность мутации
    checks["F_idempotent"] = (not spec.get("is_mutation", False)) or spec.get(
        "idempotency", None) in {"cooldown", "idempotency_key"}
    # G: узкий blast radius
    checks["G_blast_radius"] = (
        name not in DANGEROUS and not spec.get("arbitrary_exec", False))
    # H: ограниченная выдача
    checks["H_perf_limits"] = (not spec.get("returns_list", False)) or spec.get(
        "has_limit", False)

    score = sum(1 for v in checks.values() if v)
    misses = [k for k, v in checks.items() if not v]
    return {"score": score, "max": len(checks), "checks": checks, "misses": misses}


def report(title, spec):
    r = lint_tool(spec)
    print(f"{title}: score {r['score']}/{r['max']}")
    if r["misses"]:
        print("  проседает:", ", ".join(r["misses"]))
    return r

print("Линтер A->H готов: 8 проверок.")


Опишем оба наших tool как спецификации и прогоним линтер. Спецификация — это «паспорт» tool глазами линтера: не его код, а декларация свойств. `do_action` соберёт мало баллов (god-tool, свободные строки, немые ошибки), `gather` — почти максимум.

Что вы увидите: конкретные буквы, на которых проседает плохой tool. Это и есть «до» для домашних правок.


In [ ]:
spec_do_action = {
    "name": "do_action",
    "description": "Выполнить действие в игре.",      # худая заглушка
    "args": {
        "action": {"type": "str", "closed_set": True, "enum": False},   # свободная строка!
        "target": {"type": "str", "closed_set": False, "enum": False},  # свободная строка!
    },
    "free_string_action": True,
    "returns_structured": False,        # {"ok": True} / немой error
    "next_step_fields": [],
    "error_has_code": False,
    "error_has_message": False,
    "is_mutation": True,
    "idempotency": None,                # дубль на ретрае возможен (кулдаун есть, но скрыт и не объяснён)
    "arbitrary_exec": False,
    "returns_list": False,
    "has_limit": False,
}

spec_gather = {
    "name": "gather",
    "description": "Добыть ресурс с узла на своей клетке. resource — необязательное ожидание из enum {wood, stone}; без него берётся узел под ногами.",
    "args": {
        "resource": {"type": "str", "closed_set": True, "enum": True},  # enum!
        "reason":   {"type": "str", "closed_set": False, "enum": False},
    },
    "free_string_action": False,
    "returns_structured": True,         # конверт {result, cooldown, character}
    "next_step_fields": ["cooldown", "character"],
    "error_has_code": True,
    "error_has_message": True,
    "is_mutation": True,
    "idempotency": "cooldown",
    "arbitrary_exec": False,
    "returns_list": False,
    "has_limit": False,
}

print("=== ДО (god-tool) ===")
r_before = report("do_action", spec_do_action)
print()
print("=== ПОСЛЕ (split-инструмент) ===")
r_after = report("gather", spec_gather)
print()
print(f"Рост score: {r_before['score']} -> {r_after['score']} из {r_after['max']}")


## Блок 3 (опционально, нужен ключ). Живая модель против обоих наборов

До сих пор «агентом» был скрипт с зашитой стратегией — это давало воспроизводимые цифры. Теперь интереснее: дать **живой модели** оба набора и посмотреть глазами, как меняется её поведение. Модель — `MiniMax-M3` по OpenAI-совместимому протоколу (`base_url=https://api.minimax.io/v1`).

Этот блок **необязательный** и делает мягкий пропуск без ключа: если переменной `MINIMAX_API_KEY` нет (или нет сети, или нет пакета `openai`), ячейки печатают «пропущено» и идут дальше — keyless-прогон остаётся зелёным. Ключ кладите в переменную окружения или в Secrets платформы, никогда не в код.

Reasoning у M3 мы выключаем через `extra_body={"thinking": {"type": "disabled"}}` — нам нужен прямой выбор tool, а не размышления вслух.


In [ ]:
import os

MINIMAX_API_KEY = os.environ.get("MINIMAX_API_KEY")

# В Colab/Kaggle ключ часто лежит в Secrets — попробуем мягко достать.
if not MINIMAX_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        MINIMAX_API_KEY = userdata.get("MINIMAX_API_KEY")
    except Exception:
        pass
if not MINIMAX_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        MINIMAX_API_KEY = UserSecretsClient().get_secret("MINIMAX_API_KEY")
    except Exception:
        pass

LIVE_MODEL_OK = False
if not MINIMAX_API_KEY:
    print("MINIMAX_API_KEY не найден -> Блок 3 пропущен (keyless-прогон это норма).")
else:
    try:
        import openai  # noqa: F401
        LIVE_MODEL_OK = True
        print("Ключ найден, пакет openai на месте -> Блок 3 готов к запуску.")
    except Exception:
        _pip_install("openai")
        try:
            import openai  # noqa: F401
            LIVE_MODEL_OK = True
            print("Ключ найден, openai доустановлен -> Блок 3 готов.")
        except Exception as e:
            print("openai не поставился -> Блок 3 пропущен:", e)


Опишем оба набора в формате OpenAI tools (JSON Schema) и зададим модели одну и ту же задачу — «добудь дерево». На «хорошем» наборе модель почти всегда сразу зовёт `gather`/`move_dir` с правильным `enum`-аргументом. На «плохом» — пытается угадать строку в `do_action` и спотыкается о немые ошибки.

Если ключа нет — вся эта ячейка целиком пропускается одной проверкой `LIVE_MODEL_OK`.


In [ ]:
DIRECTIONS = ["north", "south", "east", "west",
              "northeast", "northwest", "southeast", "southwest"]

def good_tools_schema():
    return [
        {"type": "function", "function": {
            "name": "gather",
            "description": "Добыть ресурс с узла (дерево/камень) на своей клетке — сначала встань на него.",
            "parameters": {"type": "object", "properties": {
                "resource": {"type": "string", "enum": ["wood", "stone"],
                             "description": "Необязательное ожидание: что рассчитываешь добыть"},
                "reason": {"type": "string", "description": "Зачем (журнал намерений)"},
            }}}},
        {"type": "function", "function": {
            "name": "move_dir",
            "description": "Шаг на одну клетку в сторону direction (стороны света + диагонали).",
            "parameters": {"type": "object", "properties": {
                "direction": {"type": "string", "enum": DIRECTIONS},
                "reason": {"type": "string"},
            }, "required": ["direction"]}}},
    ]

def bad_tools_schema():
    return [
        {"type": "function", "function": {
            "name": "do_action",
            "description": "Выполнить действие в игре.",
            "parameters": {"type": "object", "properties": {
                "action": {"type": "string"},   # свободная строка
                "target": {"type": "string"},   # свободная строка
            }, "required": ["action", "target"]}}},
    ]


def ask_model(tools, user_msg):
    """Один запрос к MiniMax-M3: какой tool и с какими аргументами выберет модель."""
    from openai import OpenAI
    client = OpenAI(api_key=MINIMAX_API_KEY, base_url="https://api.minimax.io/v1")
    resp = client.chat.completions.create(
        model="MiniMax-M3",
        messages=[
            {"role": "system", "content": "Ты житель-агент в игре. Используй tools, чтобы выполнить задачу. Сразу зови нужный инструмент."},
            {"role": "user", "content": user_msg},
        ],
        tools=tools,
        tool_choice="auto",
        extra_body={"thinking": {"type": "disabled"}},   # reasoning off
    )
    msg = resp.choices[0].message
    calls = getattr(msg, "tool_calls", None) or []
    return [(c.function.name, c.function.arguments) for c in calls]


if LIVE_MODEL_OK:
    task = "Ты стоишь на клетке с деревом. Добудь древесину."
    try:
        print("ХОРОШИЙ набор -> модель выбрала:")
        for nm, ar in ask_model(good_tools_schema(), task):
            print("   ", nm, ar)
        print("ПЛОХОЙ набор -> модель выбрала:")
        for nm, ar in ask_model(bad_tools_schema(), task):
            print("   ", nm, ar)
        print()
        print("Сравните: на хорошем наборе имя tool и enum ведут модель прямо к цели;")
        print("на плохом — модель угадывает строку action/target.")
    except Exception as e:
        print("Запрос к модели не прошёл -> мягкий пропуск:", repr(e))
else:
    print("Блок 3 пропущен: нет ключа/пакета. Это ожидаемо при keyless-прогоне.")


## Блок 4 (опционально). «Хорошие» tools против живого API

Финальный мостик к реальности: наведём «хороший» интерфейс на **живой** Cognopolis по адресу `https://kindomklaster.com`. Нужен токен **твоего** жителя (в игре: Ратуша → вкладка «аккаунт» → «копировать»); он идёт в заголовке `Authorization: Bearer <token>`, и мы делаем пару настоящих действий (`move_dir`, `gather`).

Блок **необязательный** и делает мягкий пропуск, если сервера нет в сети, нет пакета `requests` или нет токена. Мы ходим **твоим** персонажем — ничего чужого не трогаем. Таймаут короткий, чтобы keyless-прогон не висел.

Мок и живой API — одна семантика: направленный шаг `POST /actions/move/{direction}`, добыча на своей клетке, тот же конверт `{result, cooldown, character}`, то же поле `banked` при шаге на дом.


In [ ]:
API_BASE = "https://kindomklaster.com"
LIVE_API_OK = False
try:
    import requests
    # быстрый ping карты (без auth) с коротким таймаутом
    _r = requests.get(f"{API_BASE}/map", timeout=4)
    LIVE_API_OK = _r.status_code == 200
    print("Живой API доступен:" if LIVE_API_OK else "API ответил, но не 200:",
          _r.status_code)
except Exception as e:
    print("Живой API недоступен -> Блок 4 пропущен (keyless-прогон это норма):", repr(e))


In [ ]:
import os
TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "")  # токен твоего жителя: Ратуша -> вкладка «аккаунт» -> «копировать»
if LIVE_API_OK and TOKEN:
    try:
        import requests, time as _t
        H = {"Authorization": f"Bearer {TOKEN}"}
        ch = requests.get(f"{API_BASE}/character", headers=H, timeout=6).json()
        print("Твой персонаж на", (ch["x"], ch["y"]), "| токен задан (скрыт)")

        # направленный шаг — координат живой API не принимает (D-069)
        direction = "east" if ch["x"] < 6 else "west"
        mv = requests.post(f"{API_BASE}/actions/move/{direction}",
                           json={"reason": "пробный шаг по живому миру"},
                           headers=H, timeout=6).json()
        print(f"move_dir({direction}) ->", mv.get("result", mv.get("error")))

        _t.sleep(mv.get("cooldown", 1.0) + 0.1)
        g = requests.post(f"{API_BASE}/actions/gather",
                          json={"reason": "пробую добыть, если стою на узле"},
                          headers=H, timeout=6).json()
        # скорее всего на этой клетке узла нет — и это тоже урок: обучающая ошибка живьём
        print("gather ->", g.get("result", g.get("error")))
        print()
        print("Видите: живой сервер отдаёт тот же конверт {result, ...} и те же ошибки")
        print("{code, message}, что мы воспроизвели в моке. Дизайн tools — боевой код.")
    except Exception as e:
        print("Запрос к живому API не прошёл -> мягкий пропуск:", repr(e))
elif LIVE_API_OK and not TOKEN:
    print("Блок 4 пропущен: задай COGNOPOLIS_TOKEN (токен жителя — Ратуша -> вкладка «аккаунт»).")
else:
    print("Блок 4 пропущен: API недоступен. Это ожидаемо при keyless-прогоне без сети.")


## Задачи

Теперь почините tools руками. Все задачи работают **на моке** из Блока 1 — ключи не нужны. Каждая задача уже содержит рабочее решение-образец (ноутбук остаётся зелёным на `Run all`); ваша работа — разобрать его, изменить под себя и прогнать снова, сверяясь с критериями приёма в конце.

Подход такой: ниже даны готовые ячейки, которые делают ровно то, что требует ДЗ. Запустите их, убедитесь, что цифры сходятся, а затем поэкспериментируйте — поменяйте имена, ошибки, значения и посмотрите, как реагируют харнесс и линтер.


### Задача 1. Переписать третий god-tool в хороший

В «плохом» наборе `do_action` прятал три действия (`move`, `gather`, `fight`). Возьмём ещё одно, которое god-tool **мог бы** проглотить, — лечение в Храме — и спроектируем его как отдельный узкий tool по канону: глагол-имя, `Pydantic`-тело, обучающие ошибки, кулдаун.

Ниже — образец. Заметьте: `heal_at_temple` уже есть в `GOOD_TOOLS` из Блока 1; здесь мы показываем, как такой tool выглядит «с нуля» рядом с его god-двойником, чтобы контраст был виден в одной ячейке. God-версия молчит и на успехе (`{"ok": true}` — сколько стоило? сколько вылечило?), и на отказе; узкая — объясняет оба случая.


In [ ]:
# --- god-двойник: лечение, спрятанное в do_action (плохо) ---
def do_action_heal(world, action, target=""):
    if action != "heal":
        return {"error": "invalid"}
    if time.monotonic() < world.busy_until:
        return {"error": "invalid"}
    missing = MAX_HP - world.hp
    cost = missing * TEMPLE_HEAL_RATE
    if missing <= 0 or world.gold < cost:
        return {"error": "invalid"}      # «уже полон» и «нет золота» неразличимы!
    world.gold -= cost
    world.hp = MAX_HP
    world.busy_until = time.monotonic() + COOLDOWN_SECONDS
    return {"ok": True}                  # сколько стоило и сколько вылечило — тайна

# --- хороший узкий tool уже написан в Блоке 1: heal_at_temple ---
# сравним оба на раненом персонаже (hp 10, лечение стоит 10 золота из 12)
w = fresh_world(); w.hp = 10
print("god   do_action_heal ->", do_action_heal(w, "heal"))
w2 = fresh_world(); w2.hp = 10
print("good  heal_at_temple ->", heal_at_temple(w2)["result"])   # {'healed': 10, 'cost': 10, 'hp': 20}

# на полном hp хороший tool объясняет (already_full_hp), плохой отвечает тем же invalid
w3 = fresh_world()
print("good  на полном hp   ->", heal_at_temple(w3))
# а без золота — отдельная ошибка с точными цифрами need/have
w4 = fresh_world(); w4.hp = 10; w4.gold = 3
print("good  без золота     ->", heal_at_temple(w4))


### Задача 2. Добавить обучающую ошибку

В Блоке 1 мы нарочно оставили одну ошибку полуслепой: `craft` на неизвестный рецепт отвечает голым `«No such recipe here.»` — код есть, а подсказки нет. Живой Cognopolis отвечает иначе: сообщение **перечисляет все крафтабельные рецепты прямо в тексте** — с ингредиентами каждого. Это «enum в форме ошибки» из лекции: даже модель, не видевшая каталога, узнаёт допустимые значения из самого отказа.

Ваш образец ниже — `craft_v2` с дословным форматом живого сервера.

:::note Почему у fight наоборот — одна ошибка на всё
Обратный приём тоже бывает осознанным: `fight` на пустой клетке и на клетке с уже побеждённым врагом отвечает **одним** `no_enemy_here` — это решение живой игры: различимый ответ стал бы оракулом таймера респауна, которым агенты фармили бы врагов. Обучающая ошибка объясняет столько, сколько **полезно** объяснить, — не больше.
:::


In [ ]:
def unknown_recipe_msg(recipe: str) -> str:
    # дословный формат живого сервера: перечислить каталог с ингредиентами
    listing = "; ".join(
        f"{name} (inputs " + ", ".join(f"{q} {r}" for r, q in spec["inputs"].items())
        + f" @ {spec['building']})"
        for name, spec in sorted(RECIPES.items()))
    return f"Unknown recipe {recipe!r}; craftable: {listing}."

def craft_v2(world: World, recipe: str, reason: Optional[str] = None) -> dict:
    """craft с обучающей ошибкой вместо полуслепой: неизвестный рецепт получает весь каталог."""
    body = CraftArgs(recipe=recipe, reason=reason)
    cd = _guard_cooldown(world)
    if cd:
        return cd
    spec = RECIPES.get(body.recipe)
    if spec is None:
        return err("unknown_recipe", unknown_recipe_msg(body.recipe))   # починили!
    need = spec["inputs"]
    if any(world.stored.get(r, 0) < q for r, q in need.items()):
        return err("not_enough_resources", not_enough_resources_msg(need, world.stored))
    for r, q in need.items():
        world.stored[r] -= q
    world.stored[body.recipe] = world.stored.get(body.recipe, 0) + 1
    return _ok(world, {"crafted": body.recipe, "spent": dict(need)})

w = fresh_world()
print("БЫЛО (Блок 1):")
print("  craft('sword') ->", craft(w, "sword")["error"])
print()
print("СТАЛО (обучающая ошибка):")
resp = craft_v2(w, "sword")
print("  craft_v2('sword') ->", resp["error"]["code"])
print("  ", resp["error"]["message"])
assert "axe_handle" in resp["error"]["message"], "листинг каталога должен быть в тексте ошибки"


### Задача 3. Сделать мутацию идемпотентной (idempotency-key)

Кулдаун — игровой приём против дубля на ретрае. В «настоящих» API ту же роль играет `idempotency-key`: клиент шлёт уникальный ключ операции, и сервер на повтор с тем же ключом отдаёт **прежний** результат, не выполняя действие заново. Реализуем это поверх `gather` — так, чтобы повтор с тем же ключом не клал второе дерево в рюкзак.

Это образец F из чек-листа в чистом виде: повторный вызов мутации не меняет мир дважды.


In [ ]:
_GATHER_SEEN = {}   # idempotency_key -> сохранённый ответ

def gather_idempotent(world: World, idempotency_key: str,
                      resource: Optional[str] = None, reason: Optional[str] = None) -> dict:
    """gather с idempotency-key: повтор с тем же ключом возвращает прежний ответ, не добывая снова."""
    if idempotency_key in _GATHER_SEEN:
        # ретрай: отдаём сохранённый результат, мир не трогаем
        cached = dict(_GATHER_SEEN[idempotency_key])
        cached["idempotent_replay"] = True
        return cached
    resp = gather(world, resource=resource, reason=reason)
    if "error" not in resp:
        _GATHER_SEEN[idempotency_key] = resp
    return resp

# встанем на дерево (1,1): из дома это один диагональный шаг
w = fresh_world()
move_dir(w, "southeast"); time.sleep(_cooldown_left(w))

key = "op-7f3a"   # уникальный ключ одной логической операции добычи
first = gather_idempotent(w, key, resource="wood")
print("первый  вызов ->", first["result"], "| рюкзак:", w.inventory)
# сеть «потеряла» ответ, фреймворк повторил с тем же ключом:
replay = gather_idempotent(w, key, resource="wood")
print("повтор  вызов ->", replay["result"], "| replay:", replay.get("idempotent_replay"),
      "| рюкзак:", w.inventory)
assert w.inventory.get("wood") == 1, "повтор с тем же ключом не должен добывать второе дерево"
print("OK: рюкзак = {'wood': 1}, дубля нет — мутация идемпотентна по ключу.")


### Задача 4. Перегнать линтер до целевого score

Соберём всё вместе: возьмём «худую» спецификацию tool и доведём её до целевого score по A→H, починив конкретные буквы. Цель — **8/8**.

Ниже две спецификации одного и того же гипотетического tool `chop_tree`: «до» (несколько проседающих букв) и «после» (всё починено). Линтер из Блока 2 покажет рост. Поэкспериментируйте: верните какую-нибудь правку обратно и посмотрите, как падает балл.


In [ ]:
# tool chop_tree «до»: имя не god-tool (A) и blast radius узкий (G), но проседают B, C, D, E, F, H
chop_before = {
    "name": "chop_tree",
    "description": "Срубить дерево.",                  # слишком коротко (<30) -> B мимо
    "args": {
        "tree": {"type": "str", "closed_set": True, "enum": False},  # свободная строка -> C мимо
    },
    "free_string_action": False,
    "returns_structured": False,        # D мимо
    "next_step_fields": [],
    "error_has_code": False,            # E мимо
    "error_has_message": False,
    "is_mutation": True,
    "idempotency": None,                # F мимо
    "arbitrary_exec": False,
    "returns_list": True,               # отдаёт список событий...
    "has_limit": False,                 # ...без лимита -> H мимо
}

# chop_tree «после»: чиним B, C, D, E, F, H
chop_after = {
    "name": "chop_tree",
    "description": ("Срубить дерево на клетке под ногами и положить древесину в рюкзак. "
                    "Зовите, стоя на клетке с деревом (дойдите move_dir); "
                    "побочный эффект — +1 wood и кулдаун."),
    "args": {
        "reason": {"type": "str", "closed_set": False, "enum": False},
    },
    "free_string_action": False,
    "returns_structured": True,
    "next_step_fields": ["cooldown", "character"],
    "error_has_code": True,
    "error_has_message": True,
    "is_mutation": True,
    "idempotency": "cooldown",
    "arbitrary_exec": False,
    "returns_list": True,
    "has_limit": True,                  # limit на выдачу событий
}

print("=== chop_tree ДО ===")
rb = report("chop_tree (before)", chop_before)
print()
print("=== chop_tree ПОСЛЕ ===")
ra = report("chop_tree (after)", chop_after)
print()
TARGET = 8
print(f"score: {rb['score']} -> {ra['score']} (цель {TARGET}/8)")
assert ra["score"] >= TARGET, "после правок tool должен набрать целевой score 8/8"
print("Цель достигнута: tool прошёл весь чек-лист A->H.")


## Что дальше

Вы прошли путь от «тупящего» агента до починенного интерфейса: расщепили god-tool, поставили `enum`-рельсы, написали обучающие ошибки, сделали мутацию идемпотентной и прогнали линтер по A→H. Эта же линза работает на любом tool — своём или чужом.

Следующий примитив — тот же инструмент, но описанный словами и положенный в память агента: [Модуль 13.6. Skills](https://itrubnikov.github.io/Train_of_Thought/docs/modules/13-6-skills/). А затем [Модуль 14.5. MCP](https://itrubnikov.github.io/Train_of_Thought/docs/modules/14-5-mcp/) — как один контракт инструмента отдать по стандартному протоколу любому агенту.

**Критерии приёма (проверяете сами):**

- ноутбук прогнан целиком (`Run all`) keyless без ошибок;
- god-tool переписан в узкий инструмент по одной цели с обучающими ошибками (Задача 1);
- полуслепая ошибка `unknown_recipe` превращена в обучающую с листингом каталога (Задача 2);
- мутация сделана идемпотентной — повтор не дублирует эффект (Задача 3);
- линтер по A→H показывает рост score «до/после» и достигает цели 8/8 (Задача 4).

Если так — домашка сдана, преподаватель не нужен.
